In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

import sys
sys.path.append('./src')
import warnings
warnings.filterwarnings("ignore")

from data import PPCI

In [ ]:
splitting_criterias = ['all','random_easy','experiment1','position','position_easy','random', 'treatment0', 'treatment1', 'treatment2','experiment','experiment1']
splitting_criterias = ["all","experiment1", "position_easy","position","treatment0","treatment2"]
exp = "tar"
task = "or"
filter="bias_ref"
colors = ['#6DAEDB', '#A8D5BA', '#F2CAC8', '#D3CCE3', '#FFD580']#['blue', 'green', 'red', 'purple', 'orange']
for splitting_criteria in splitting_criterias:
    print(splitting_criteria)
    results = pd.read_csv(f'results/generalization/{task}/{splitting_criteria}.csv')
    results["bias_ref"] = abs(results["PPATE_ref"] - results["ATE_ref"])
    results["bias_tar"] = abs(results["PPATE_tar"] - results["ATE_tar"])
    results = results.loc[results.groupby(['method', 'encoder'])[filter].idxmin()]
    results["encoder"] = results["encoder"].replace({
        "vit_large": "ViT-L",
        "vit": "ViT-S",
        "clip_large": "CLIP-ViT-L",
        "clip": "CLIP-ViT-S",
        "mae": "MAE",
        "dino": "DINOv2"})
    # order first DINO then MAE then CLIP then ViT
    results["encoder"] = pd.Categorical(results["encoder"], ["DINOv2", "MAE", "CLIP-ViT-L", "CLIP-ViT-S", "ViT-L", "ViT-S"])
    results['bias_ref'] = results['PPATE_ref'] - results['ATE_ref']
    results['bias_tar'] = results['PPATE_tar'] - results['ATE_tar']
    results["detection_ref"] = results["PPATE_p_value_ref"] < 0.05
    results["detection_tar"] = results["PPATE_p_value_tar"] < 0.05
    print(results.groupby(['method']).agg({'bias_ref': 'mean', 'bias_tar': 'mean', 'detection_ref': 'mean',  'detection_tar': 'mean'}).reset_index())
    print(results.groupby(['method']).agg({'acc_ref': 'mean', 'acc_tar': 'mean'}).reset_index())
    print(results.groupby(['method']).agg({'bacc_ref': 'mean', 'bacc_tar': 'mean'}).reset_index())
    methods = ["DERM","vREx","ERM"]
    results = results.sort_values(by=['method', 'encoder'])
    fig, ax = plt.subplots(figsize=(8, 5))
    for k, encoder in enumerate(results['encoder'].unique()[::-1]):
        ax.plot([], [], 'o', color=colors[len(results['encoder'].unique())-k-1], label=f'{encoder}')
    ax.legend(loc='upper left')
    for i, method in enumerate(methods):
        subset = results[results['method'] == method]
        means = subset[f'PPATE_{exp}']
        errors = subset[f'PPATE_std_{exp}']*1.96
        encoders = subset['encoder']
        for j, (mean, std) in enumerate(zip(means, errors)):
            ax.errorbar(mean, i + j * 0.15, xerr=std, fmt='o', label=None if j else method, capsize=5, color=colors[j])
    ax.errorbar(results[f'ATE_{exp}'].iloc[0], i + j * 0.15+0.5, xerr=subset[f'ATE_std_{exp}'].iloc[0]*1.96, fmt='o', label="Ground Truth", capsize=5, color='black')
    ax.set_yticks(list(np.arange(len(methods)) + 0.3)+[i + j * 0.15+0.5], labels=methods + ["Ground Truth"])
    ax.set_xlabel(r'$\hat{\tau}$')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.axvline(x=results[f'ATE_{exp}'].mean(), color='black', linestyle='--', label=f'ATE_{exp}')
    plt.show()
    fig.savefig(f'results/generalization/{task}/{splitting_criteria}_{exp}_{filter}.png', dpi=300)

In [ ]:
split_criteria = "all"
reference = PPCI(encoder = "dino",
               token = "class",
               task = "sum",
               split_criteria = split_criteria,
               environment = "supervised",
               batch_size = 256,
               num_proc = 4,
               verbose = True,
               data_dir = 'data/istant_lq',
               results_dir = 'results/istant_lq')
target = PPCI(encoder = "dino",
               token = "class",
               task = "sum",
               split_criteria = split_criteria,
               environment = "supervised",
               batch_size = 256,
               num_proc = 4,
               verbose = True,
               data_dir = 'data/istant_hq',
               results_dir = 'results/istant_hq')

In [ ]:
reference.train(add_pred_env="supervised", 
                hidden_layers = 2,
                hidden_nodes = 256,
                batch_size = 256,
                lr = 0.0005,
                seed = 0,
                num_epochs=1,
                save = False,
                verbose = True,
                force = False,
                method = "vREx")

In [ ]:
reference.supervised["Y_hat"].max()

In [ ]:
results = pd.DataFrame(columns=['method','seed','ATE_ref', 'ATE_std_ref', 'PPATE_ref', 'PPATE_std_ref', 'ATE_target', 'ATE_std_target', 'PPATE_target', 'PPATE_std_target', 'acc_ref', 'bacc_ref', 'acc_target', 'bacc_target'])
i = 0
for method in ["ERM","vREx","DERM"]:
    for seed in range(5):
        print(f"Method: {method}, Seed: {seed}")
        reference.train(add_pred_env="supervised", 
                hidden_layers = 2,
                hidden_nodes = 256,
                batch_size = 256,
                lr = 0.0005,
                seed = seed,
                num_epochs=15,
                save = False,
                verbose = True,
                force = False,
                method = method)
        target.results_dir = 'results/istant_lq'
        target.train(add_pred_env="supervised", 
                    hidden_layers = 2,
                    hidden_nodes = 256,
                    batch_size = 256,
                    lr = 0.0005,
                    seed = seed,
                    num_epochs = 10,
                    save = False,
                    verbose=True,
                    method = method)
        target.results_dir = 'results/istant_hq'
        i += 1
        results.loc[i] = [method, seed, ATE_ref, ATE_std_ref, PPATE_ref, PPATE_std_ref, ATE_target, ATE_std_target, PPATE_target, PPATE_std_target, acc_ref, bacc_ref, acc_target, bacc_target]
results.to_csv(f"results/{split_criteria}_generalization.csv", index=False)
results